In [2]:
from wsfr_download.config import DATA_ROOT
print(DATA_ROOT)

ModuleNotFoundError: No module named 'wsfr_download'

In [2]:
"""Code for downloading the SNODAS snowpack property estimates from data assimilation. Use the CLI
to download, for example:

    python -m wsfr_download snodas 2005 2007 2009

or use the `bulk` command to download many sources at once based on a config file.

    python -m wsfr_download bulk data_download/hindcast_test_config.yml

You can also import this module and use it as a library.

See the challenge website for more about this approved data source:
https://www.drivendata.org/competitions/254/reclamation-water-supply-forecast-dev/page/801/#snodas

See additional references:

Reference for downloading data from online directory
https://nsidc.org/data/user-resources/help-center/how-access-and-download-noaansidc-data

Reference for converting .dat files to netcdf
https://nsidc.org/data/user-resources/help-center/how-do-i-convert-snodas-binary-files-geotiff-or-netcdf

User guide:
https://nsidc.org/sites/default/files/g02158-v001-userguide_2_1.pdf
"""
import calendar
from datetime import datetime, timedelta
import functools
from pathlib import Path
from typing import Annotated

#from loguru import logger
import pandas as pd
import requests
from tqdm.contrib.concurrent import thread_map
import typer

#from wsfr_download.config import DATA_ROOT
#from wsfr_download.utils import DownloadResult, log_download_results


def urls_for_date_range(start_date: datetime, end_date: datetime) -> list[str]:
    """Get all source urls for files within a given date range (inclusive).

    Args:
        start_date (datetime): Start date
        end_date (datetime): End date
    """
    dates_in_range = pd.date_range(start_date, end_date, freq="1D")

    return [date_url(date) for date in dates_in_range]


def date_url(date: datetime) -> str:
    """Get the source url for the SNODAS file on a given date"""
    url = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/"
    url += f"{date.year}/{date.month:02}_{calendar.month_abbr[date.month]}"
    url += f"/SNODAS_{date.strftime('%Y%m%d')}.tar"

    return url


def download_from_url(source_url: str, out_dir: Path, skip_existing: bool) -> DownloadResult:
    """Download a SNODAS file based on its source URL. The file will be saved to
    out_dir with the same filename as the URL.
    """
    filename = source_url.split("/")[-1]
    out_file = out_dir / filename
    if skip_existing and out_file.exists():
        return DownloadResult.SKIPPED_EXISTING

    response = requests.get(source_url)
    with open(out_file, "wb") as f:
        f.write(response.content)

    return DownloadResult.SUCCESS


#def download_snodas(
    #forecast_years: Annotated[list[int], typer.Argument(help="Forecast years to download for.")],
    #fy_start_month: Annotated[int, typer.Option(help="Forecast year start month.")] = 10,
    #fy_start_day: Annotated[int, typer.Option(help="Forecast year start day.")] = 1,
    #fy_end_month: Annotated[int, typer.Option(help="Forecast year end month.")] = 7,
    #fy_end_day: Annotated[int, typer.Option(help="Forecast year end day.")] = 22,
    #skip_existing: Annotated[bool, typer.Option(help="Whether to skip an existing file.")] = True,
#):
#    """Download SNODAS data from NOAA NSIDC:
#    https://noaadata.apps.nsidc.org/NOAA/G02158/masked/
#    Each forecast year begins on the specified date of the previous calendar
#    year, and ends on the specified date of the current calendar year. By
#    default, each forecast year starts on October 1 and ends July 21; e.g.,
#    by default, FY2021 starts on 2020-10-01 and ends on 2021-07-21.

#    When provided, the end date defined by `fy_end_month` and `fy_end_day` is exclusive,
#    i.e. for July 22, the most recent data will be for July 21."""
#    logger.info("Downloading SNODAS data...")
#
#    all_download_results = []
#    for fy in forecast_years:
#        # Get all URLs to download in given forecast year
#        fy_start = datetime(fy - 1, fy_start_month, fy_start_day)
#        end_date = datetime(fy, fy_end_month, fy_end_day)
#        fy_end = end_date - timedelta(days=1)
#        fy_urls = urls_for_date_range(fy_start, fy_end)

#        # Download forecast year files
#        out_dir = DATA_ROOT / f"snodas/FY{str(fy)}"
#        out_dir.mkdir(exist_ok=True, parents=True)

#        logger.info(
#            f"Downloading {len(fy_urls)} files for forecast year {fy}"
#            f" ({fy_start.strftime('%Y-%m-%d')} to {fy_end.strftime('%Y-%m-%d')})"
#        )
#        fy_download_results = thread_map(
#            functools.partial(download_from_url, out_dir=out_dir, skip_existing=skip_existing),
#            fy_urls,
#            total=len(fy_urls),
#            chunksize=1,
#        )
#        all_download_results += fy_download_results

#    log_download_results(all_download_results)
#    logger.success("SNODAS download complete.")


#if __name__ == "__main__":
#    typer.run(download_snodas)

NameError: name 'DownloadResult' is not defined

In [10]:
#Converting .dat to .tif
import glob, os, rasterio


folderr = r'data'
dat_file = os.path.join(folderr, 'us_ssmv11034tS__T0001TTNATS2004040105HP001.dat')
hdr_file = os.path.join(folderr, 'us_ssmv11034tS__T0001TTNATS2004040105HP001.hdr') #I made this file based on the user guide.

from rasterio.transform import from_origin

# 1. Define your known coordinates (EPSG:4326)
ul_lat, ul_lon = 52.87500000000000, -124.73333333333333  # Upper Left -124.73333333333333 52.87500000000000 #Got from user guide
lr_lat, lr_lon = 24.95000000000000, -66.94166666666667  # Lower Right -66.94166666666667 24.95000000000000 #got from user guide

# 2. Open the file to get dimensions
with rasterio.open(dat_file) as src:
    data = src.read()
    width = src.width
    height = src.height

# 3. Calculate pixel resolution
# Note: x_res is usually positive, y_res is usually negative
# because latitudes decrease as you go down the image.
res_x = (lr_lon - ul_lon) / width
res_y = (ul_lat - lr_lat) / height

# 4. Create the transform
# from_origin(west, north, xsize, ysize)
new_transform = from_origin(ul_lon, ul_lat, res_x, res_y)

# 5. Define the CRS
new_crs = 'EPSG:4326'

# 6. (Optional) Save to a new GeoTIFF with the correct metadata
with rasterio.open(
    os.path.join(folderr, 'test_binary_prj.tif'), #Change this outputname in your loop
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=src.count,
    dtype=data.dtype,
    crs=new_crs,
    transform=new_transform,
) as dst:
    dst.write(data)

/mnt/shared/moduleapps/EarthSci/Python-3.9.15-Fixed/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [2]:
#full .dat to tif. pipeline:

import os, glob, tarfile, gzip, shutil, calendar, requests
import rasterio
from rasterio.transform import from_origin
from datetime import datetime
from pathlib import Path
import pandas as pd

# --- Directories ---
data_dir = Path('data/SNODAS_SWE_20042024')
data_dir.mkdir(exist_ok=True, parents=True)

# --- Spatial parameters ---
ul_lat, ul_lon = 52.87500000000000, -124.73333333333333
lr_lat, lr_lon = 24.95000000000000, -66.94166666666667
nrows, ncols = 3351, 6935
res_x = (lr_lon - ul_lon) / ncols
res_y = (ul_lat - lr_lat) / nrows
transform = from_origin(ul_lon, ul_lat, res_x, res_y)
crs = 'EPSG:4326'
nodata = -9999

# --- HDR content (ENVI format) ---
hdr_content = """ENVI
samples = 6935
lines = 3351
bands = 1
header offset = 0
file type = ENVI Standard
data type = 2
interleave = bsq
byte order = 1
"""

# --- URL builders ---
def date_url(date: datetime) -> str:
    url = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/"
    url += f"{date.year}/{date.month:02}_{calendar.month_abbr[date.month]}"
    url += f"/SNODAS_{date.strftime('%Y%m%d')}.tar"
    return url

def urls_for_date_range(start_date: datetime, end_date: datetime) -> list:
    dates = pd.date_range(start_date, end_date, freq='1D')
    return [date_url(date) for date in dates]

# --- Main pipeline ---
def process_date(url: str, skip_existing: bool = True):
    filename = url.split('/')[-1]                        # SNODAS_yyyymmdd.tar
    tar_path = data_dir / filename

    # 1. Download tar
    if skip_existing and tar_path.exists():
        print(f'Skipping download: {filename}')
    else:
        print(f'Downloading: {filename}')
        response = requests.get(url)
        with open(tar_path, 'wb') as f:
            f.write(response.content)

    # 2. Extract only the 11034 .dat.gz from the tar
    with tarfile.open(tar_path, 'r') as tar:
        swe_member = next(
            (m for m in tar.getmembers() if '11034' in m.name and m.name.endswith('.dat.gz')),
            None
        )
        if swe_member is None:
            print(f'No SWE file found in {filename}')
            return
        tar.extract(swe_member, path=data_dir)

    dat_gz_path = data_dir / swe_member.name

    # 3. Decompress .dat.gz to .dat
    dat_path = dat_gz_path.with_suffix('')  # strips .gz
    with gzip.open(dat_gz_path, 'rb') as gz_in:
        with open(dat_path, 'wb') as dat_out:
            shutil.copyfileobj(gz_in, dat_out)

    # 4. Write matching .hdr so rasterio can open the .dat
    hdr_path = dat_path.with_suffix('.hdr')
    with open(hdr_path, 'w') as f:
        f.write(hdr_content)

    # 5. Convert .dat to .tif
    tif_path = dat_path.with_suffix('.tif')
    with rasterio.open(dat_path) as src:
        data = src.read()
        width = src.width
        height = src.height

    with rasterio.open(
        tif_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=data.dtype,
        crs=crs, transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(data)

    # 6. Clean up intermediates
    os.remove(dat_gz_path)
    os.remove(dat_path)
    os.remove(hdr_path)
    os.remove(tar_path)

    print(f'Written: {tif_path.name}')

# --- Run for date range ---
start = datetime(2004, 3, 1)
end   = datetime(2004, 3, 7)

urls = urls_for_date_range(start, end)
for url in urls:
    process_date(url)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Downloading: SNODAS_20040301.tar


/mnt/shared/moduleapps/EarthSci/Python-3.9.15-Fixed/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Written: us_ssmv11034tS__T0001TTNATS2004030105HP001.tif
Downloading: SNODAS_20040302.tar
Written: us_ssmv11034tS__T0001TTNATS2004030205HP001.tif
Downloading: SNODAS_20040303.tar
Written: us_ssmv11034tS__T0001TTNATS2004030305HP001.tif
Downloading: SNODAS_20040304.tar
Written: us_ssmv11034tS__T0001TTNATS2004030405HP001.tif
Downloading: SNODAS_20040305.tar
Written: us_ssmv11034tS__T0001TTNATS2004030505HP001.tif
Downloading: SNODAS_20040306.tar
Written: us_ssmv11034tS__T0001TTNATS2004030605HP001.tif
Downloading: SNODAS_20040307.tar
Written: us_ssmv11034tS__T0001TTNATS2004030705HP001.tif


In [3]:
#Shower Falls SWE pipeline
import rasterstats
import pandas as pd
from shapely.geometry import mapping, Point
import glob, os, re

folderr = r'data/SNODAS_SWE_20042024'
lat = 45.4
lon = -110.95
point_geom = mapping(Point(lon, lat))

tif_files = glob.glob(os.path.join(folderr, '*.tif'))

rows = []

for tif_file in tif_files:
    filename = os.path.splitext(os.path.basename(tif_file))[0]
    
    match = re.search(r'TS(\d{8})', filename)
    if not match:
        print(f'Skipping {filename} — no date found')
        continue
    date = pd.to_datetime(match.group(1), format='%Y%m%d')
    
    result = rasterstats.point_query(
        [point_geom],
        tif_file,
        nodata=-9999,
        interpolate='nearest'
    )[0]
    
    swe = result / 1000 if result is not None else None
    rows.append({'date': date, 'swe_m': swe})

df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)

out_csv = os.path.join(folderr, 'shower_falls_swe.csv')
df.to_csv(out_csv, index=False)

print(f'Done. {len(df)} records written to {out_csv}')
print(df.head())

Done. 7 records written to data/SNODAS_SWE_20042024/shower_falls_swe.csv
        date  swe_m
0 2004-03-01  0.325
1 2004-03-02  0.353
2 2004-03-03  0.353
3 2004-03-04  0.353
4 2004-03-05  0.369


In [5]:
#FULL PIPELINE (URL --> SWE)
import os, tarfile, gzip, shutil, calendar, requests, re, time
import rasterio
import rasterstats
from rasterio.transform import from_origin
from datetime import datetime
from pathlib import Path
from shapely.geometry import mapping, Point
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# --- Directories ---
data_dir = Path('data/SNODAS_SWE_20042024')
data_dir.mkdir(exist_ok=True, parents=True)

# --- Spatial parameters ---
ul_lat, ul_lon = 52.87500000000000, -124.73333333333333
lr_lat, lr_lon = 24.95000000000000, -66.94166666666667
nrows, ncols = 3351, 6935
res_x = (lr_lon - ul_lon) / ncols
res_y = (ul_lat - lr_lat) / nrows
transform = from_origin(ul_lon, ul_lat, res_x, res_y)
crs = 'EPSG:4326'
nodata = -9999

# --- HDR content (ENVI format) ---
hdr_content = """ENVI
samples = 6935
lines = 3351
bands = 1
header offset = 0
file type = ENVI Standard
data type = 2
interleave = bsq
byte order = 1
"""

# --- Shower Falls point ---
lat = 45.4
lon = -110.95
point_geom = mapping(Point(lon, lat))

# --- URL builders ---
def date_url(date: datetime) -> str:
    url = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/"
    url += f"{date.year}/{date.month:02}_{calendar.month_abbr[date.month]}"
    url += f"/SNODAS_{date.strftime('%Y%m%d')}.tar"
    return url

def urls_for_date_range(start_date: datetime, end_date: datetime) -> list:
    dates = pd.date_range(start_date, end_date, freq='1D')
    return [date_url(date) for date in dates]

# --- Network Retry Loop  ---
class DownloadFailedError(Exception):
    pass

def download_with_retry(url: str, out_path: Path, max_retries: int = 5):
    time.sleep(1) #small delay
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=60)
            with open(out_path, 'wb') as f:
                f.write(response.content)
            return True
        except Exception as e:
            print(f'Attempt {attempt} failed: {e}')
            if out_path.exists():
                os.remove(out_path)
            time.sleep(5 * attempt)
    raise DownloadFailedError(f'Failed after {max_retries} attempts: {url}')

# --- Main pipeline ---
# Thread-safe list for collecting rows
rows_lock = threading.Lock()
rows = []

def process_date(url: str) -> dict:
    filename = url.split('/')[-1]
    tar_path = data_dir / filename

    # 1. Download tar
    print(f'Downloading: {filename}')
    success = download_with_retry(url, tar_path)
    if not success:
        return None

    # 2. Extract 11034 .dat.gz
    with tarfile.open(tar_path, 'r') as tar:
        swe_member = next(
            (m for m in tar.getmembers() if '11034' in m.name and m.name.endswith('.dat.gz')),
            None
        )
        if swe_member is None:
            print(f'No SWE file found in {filename}')
            os.remove(tar_path)
            return None
        tar.extract(swe_member, path=data_dir)

    dat_gz_path = data_dir / swe_member.name

    # 3. Decompress .dat.gz to .dat
    dat_path = dat_gz_path.with_suffix('')
    with gzip.open(dat_gz_path, 'rb') as gz_in:
        with open(dat_path, 'wb') as dat_out:
            shutil.copyfileobj(gz_in, dat_out)

    # 4. Write .hdr
    hdr_path = dat_path.with_suffix('.hdr')
    with open(hdr_path, 'w') as f:
        f.write(hdr_content)

    # 5. Convert .dat to .tif
    tif_path = dat_path.with_suffix('.tif')
    with rasterio.open(dat_path) as src:
        data = src.read()
        width = src.width
        height = src.height
    with rasterio.open(
        tif_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=data.dtype,
        crs=crs, transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(data)

    # 6. Query SWE at Shower Falls
    match = re.search(r'TS(\d{8})', dat_path.stem)
    date = pd.to_datetime(match.group(1), format='%Y%m%d')
    result = rasterstats.point_query(
        [point_geom],
        str(tif_path),
        nodata=-9999,
        interpolate='nearest'
    )[0]
    swe = result / 1000 if result is not None else None

    # 7. Delete all intermediates including .tif
    for f in [tar_path, dat_gz_path, dat_path, hdr_path, tif_path]:
        if f.exists():
            os.remove(f)

    print(f'Queried: {dat_path.stem} — SWE: {swe} m')
    return {'date': date, 'swe_m': swe}

# --- Run for date range ---
urls = []
for year in range(2004, 2005):
    year_urls = urls_for_date_range(
        datetime(year, 3, 1),
        datetime(year, 6, 30)
    )
    urls.extend(year_urls)
    
start_time = time.time()
rows = []
failed_url = None

try:
    with ThreadPoolExecutor(max_workers=1) as executor:
        futures = {executor.submit(process_date, url): url for url in urls}
        for future in as_completed(futures):
            try:
                result = future.result()
                if result is not None:
                    rows.append(result)
            except DownloadFailedError as e:
                failed_url = futures[future]
                print(f'Download failed, stopping: {failed_url}')
                executor.shutdown(wait=False, cancel_futures=True)
                break
finally:
    # Always save whatever was collected
    if rows:
        df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
        out_csv = data_dir / 'SNODAS_swe_2004.csv'
        if out_csv.exists():
            existing_df = pd.read_csv(out_csv, parse_dates=['date'])
            df = pd.concat([existing_df, df]).sort_values('date').reset_index(drop=True)
        df.to_csv(out_csv, index=False)
        elapsed = time.time() - start_time
        print(f'Saved {len(df)} total records to {out_csv}')
        print(f'Elapsed time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
        if failed_url:
            print(f'Stopped at: {failed_url}')

Downloading: SNODAS_20040301.tar
Attempt 1 failed: HTTPSConnectionPool(host='noaadata.apps.nsidc.org', port=443): Max retries exceeded with url: /NOAA/G02158/masked/2004/03_Mar/SNODAS_20040301.tar (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdcd30a5810>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 2 failed: HTTPSConnectionPool(host='noaadata.apps.nsidc.org', port=443): Max retries exceeded with url: /NOAA/G02158/masked/2004/03_Mar/SNODAS_20040301.tar (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdcd2558550>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 3 failed: HTTPSConnectionPool(host='noaadata.apps.nsidc.org', port=443): Max retries exceeded with url: /NOAA/G02158/masked/2004/03_Mar/SNODAS_20040301.tar (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdcd23e5b90>: Failed to establish a new connection: [Errno 111]

NameError: name 'df' is not defined

In [5]:
#FULL PIPELINE (URL --> SWE)
import os, tarfile, gzip, shutil, calendar, requests, re, time
import rasterio
import rasterstats
from rasterio.transform import from_origin
from datetime import datetime
from pathlib import Path
from shapely.geometry import mapping, Point
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# --- Directories ---
data_dir = Path(r'C:\Users\jonas\OneDrive\Documents\MSU\2026 Spring\Data Science\Final Project\Data\SNODAS')
data_dir.mkdir(exist_ok=True, parents=True)

# --- Spatial parameters ---
ul_lat, ul_lon = 52.87500000000000, -124.73333333333333
lr_lat, lr_lon = 24.95000000000000, -66.94166666666667
nrows, ncols = 3351, 6935
res_x = (lr_lon - ul_lon) / ncols
res_y = (ul_lat - lr_lat) / nrows
transform = from_origin(ul_lon, ul_lat, res_x, res_y)
crs = 'EPSG:4326'
nodata = -9999

# --- HDR content (ENVI format) ---
hdr_content = """ENVI
samples = 6935
lines = 3351
bands = 1
header offset = 0
file type = ENVI Standard
data type = 2
interleave = bsq
byte order = 1
"""

# --- Shower Falls point ---
lat = 45.4
lon = -110.95
point_geom = mapping(Point(lon, lat))

# --- URL builders ---
def date_url(date: datetime) -> str:
    url = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/"
    url += f"{date.year}/{date.month:02}_{calendar.month_abbr[date.month]}"
    url += f"/SNODAS_{date.strftime('%Y%m%d')}.tar"
    return url

def urls_for_date_range(start_date: datetime, end_date: datetime) -> list:
    dates = pd.date_range(start_date, end_date, freq='1D')
    return [date_url(date) for date in dates]

# --- Network Retry Loop  ---
class DownloadFailedError(Exception):
    pass

def download_with_retry(url: str, out_path: Path, max_retries: int = 5):
    time.sleep(5) #small delay
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=60)
            with open(out_path, 'wb') as f:
                f.write(response.content)
            return True
        except Exception as e:
            print(f'Attempt {attempt} failed: {e}')
            if out_path.exists():
                os.remove(out_path)
            time.sleep(5 * attempt)
    raise DownloadFailedError(f'Failed after {max_retries} attempts: {url}')

# --- Main pipeline ---
# Thread-safe list for collecting rows
rows_lock = threading.Lock()
rows = []

def process_date(url: str) -> dict:
    filename = url.split('/')[-1]
    tar_path = data_dir / filename

    # 1. Download tar
    print(f'Downloading: {filename}')
    success = download_with_retry(url, tar_path)
    if not success:
        return None

    # 2. Extract 11034 .dat.gz
    try:
        with tarfile.open(tar_path, 'r') as tar:
            swe_member = next(
                (m for m in tar.getmembers() if '11034' in m.name and m.name.endswith('.dat.gz')),
                None
            )
            if swe_member is None:
                print(f'No SWE file found in {filename}')
                os.remove(tar_path)
                return None
            tar.extract(swe_member, path=data_dir)
    except tarfile.ReadError:
        print(f'Corrupt tar file, skipping: {filename}')
        if tar_path.exists():
            os.remove(tar_path)
        return None

    dat_gz_path = data_dir / swe_member.name

    # 3. Decompress .dat.gz to .dat
    dat_path = dat_gz_path.with_suffix('')
    with gzip.open(dat_gz_path, 'rb') as gz_in:
        with open(dat_path, 'wb') as dat_out:
            shutil.copyfileobj(gz_in, dat_out)

    # 4. Write .hdr
    hdr_path = dat_path.with_suffix('.hdr')
    with open(hdr_path, 'w') as f:
        f.write(hdr_content)

    # 5. Convert .dat to .tif
    tif_path = dat_path.with_suffix('.tif')
    with rasterio.open(dat_path) as src:
        data = src.read()
        width = src.width
        height = src.height
    with rasterio.open(
        tif_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=data.dtype,
        crs=crs, transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(data)

    # 6. Query SWE at Shower Falls
    match = re.search(r'TS(\d{8})', dat_path.stem)
    date = pd.to_datetime(match.group(1), format='%Y%m%d')
    result = rasterstats.point_query(
        [point_geom],
        str(tif_path),
        nodata=-9999,
        interpolate='nearest'
    )[0]
    swe = result / 1000 if result is not None else None

    # 7. Delete all intermediates including .tif
    for f in [tar_path, dat_gz_path, dat_path, hdr_path, tif_path]:
        if f.exists():
            os.remove(f)

    print(f'Queried: {dat_path.stem} — SWE: {swe} m')
    return {'date': date, 'swe_m': swe}

# --- Run for missing dates ---
missing_dates = [
    datetime(2005, 6, 25), datetime(2005, 6, 26), datetime(2005, 6, 27),
    datetime(2005, 6, 28), datetime(2005, 6, 29), datetime(2005, 6, 30),
    datetime(2007, 3, 26), datetime(2008, 3, 13),
    datetime(2008, 6, 13), datetime(2008, 6, 18),
]

urls = [date_url(d) for d in missing_dates]

start_time = time.time()
rows = []
failed_url = None

try:
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(process_date, url): url for url in urls}
        for future in as_completed(futures):
            try:
                result = future.result()
                if result is not None:
                    rows.append(result)
            except DownloadFailedError as e:
                failed_url = futures[future]
                print(f'Download failed, stopping: {failed_url}')
                executor.shutdown(wait=False, cancel_futures=True)
                break
finally:
    if rows:
        df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
        out_csv = data_dir / 'SNODAS_swe_missingdates.csv'
        if out_csv.exists():
            existing_df = pd.read_csv(out_csv, parse_dates=['date'])
            df = pd.concat([existing_df, df]).sort_values('date').reset_index(drop=True)
        df.to_csv(out_csv, index=False)
        elapsed = time.time() - start_time
        print(f'Saved {len(df)} total records to {out_csv}')
        print(f'Elapsed time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
        if failed_url:
            print(f'Stopped at: {failed_url}')

Downloading: SNODAS_20050625.tar
Downloading: SNODAS_20050626.tar
Corrupt tar file, skipping: SNODAS_20050625.tar
Downloading: SNODAS_20050627.tar


C:\Users\jonas\miniconda3\envs\gphy471\lib\site-packages\rasterio\__init__.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Queried: us_ssmv11034tS__T0001TTNATS2005062605HP001 — SWE: 0.0 m
Downloading: SNODAS_20050628.tar
Queried: us_ssmv11034tS__T0001TTNATS2005062705HP001 — SWE: 0.0 m
Downloading: SNODAS_20050629.tar
Queried: us_ssmv11034tS__T0001TTNATS2005062805HP001 — SWE: 0.0 m
Downloading: SNODAS_20050630.tar
Queried: us_ssmv11034tS__T0001TTNATS2005062905HP001 — SWE: 0.0 m
Downloading: SNODAS_20070326.tar
Queried: us_ssmv11034tS__T0001TTNATS2005063005HP001 — SWE: 0.0 m
Downloading: SNODAS_20080313.tar
Corrupt tar file, skipping: SNODAS_20070326.tar
Downloading: SNODAS_20080613.tar
Corrupt tar file, skipping: SNODAS_20080313.tar
Downloading: SNODAS_20080618.tar
Corrupt tar file, skipping: SNODAS_20080613.tar
Corrupt tar file, skipping: SNODAS_20080618.tar
Saved 5 total records to C:\Users\jonas\OneDrive\Documents\MSU\2026 Spring\Data Science\Final Project\Data\SNODAS\SNODAS_swe_missingdates.csv
Elapsed time: 29.6 seconds (0.5 minutes)


In [2]:
#Combine SNODAS data into 1 .csv
import pandas as pd
import glob
import os

# Load all SNODAS yearly CSVs and combine
snodas_files = glob.glob(r'C:\Users\jonas\OneDrive\Documents\MSU\2026 Spring\Data Science\Final Project\Data\SNODAS\*.csv')
snodas_df = pd.concat([pd.read_csv(f, parse_dates=['date']) for f in snodas_files])
snodas_df = snodas_df.sort_values('date').reset_index(drop=True)
snodas_df = snodas_df.rename(columns={'swe_m': 'swe_m_snodas'})

print(f'{len(snodas_df)} total records')
print(snodas_df.head())

# Save
snodas_df.to_csv(r'C:\Users\jonas\OneDrive\Documents\MSU\2026 Spring\Data Science\Final Project\Data\SNODAS\SNODAS_swe_all.csv', index=False)

2552 total records
        date  swe_m_snodas
0 2004-03-01         0.325
1 2004-03-02         0.353
2 2004-03-03         0.353
3 2004-03-04         0.353
4 2004-03-05         0.369


In [1]:
# Pixel dimensions
import numpy as np

# Known SNODAS grid parameters
ul_lon = -124.73333333333333
ul_lat  = 52.87500000000000
res_x   = ((-66.94166666666667) - ul_lon) / 6935  # degrees per pixel in x
res_y   = (52.87500000000000 - 24.95000000000000) / 3351  # degrees per pixel in y

# Shower Falls point
point_lon = -110.95
point_lat  = 45.4

# Find which pixel the point falls in
col = int((point_lon - ul_lon) / res_x)
row = int((ul_lat - point_lat) / res_y)

# Calculate pixel corner coordinates
pixel_west  = ul_lon + col * res_x
pixel_east  = pixel_west + res_x
pixel_north = ul_lat - row * res_y
pixel_south = pixel_north - res_y

print(f'Pixel column: {col}, row: {row}')
print(f'West:  {pixel_west:.6f}')
print(f'East:  {pixel_east:.6f}')
print(f'North: {pixel_north:.6f}')
print(f'South: {pixel_south:.6f}')
print(f'Resolution: {res_x*111:.2f} km (x), {res_y*111:.2f} km (y)')

Pixel column: 1654, row: 897
West:  -110.950000
East:  -110.941667
North: 45.400000
South: 45.391667
Resolution: 0.92 km (x), 0.92 km (y)


In [2]:
import geopandas as gpd
from shapely.geometry import box, Point
import os

os.makedirs(r'RESULTS\spatial', exist_ok=True)

# --- SNODAS pixel as a polygon ---
pixel_polygon = box(pixel_west, pixel_south, pixel_east, pixel_north)
pixel_gdf = gpd.GeoDataFrame(
    {'name': ['SNODAS_pixel'], 'geometry': [pixel_polygon]},
    crs='EPSG:4326'
)
pixel_gdf.to_file(r'RESULTS\spatial\snodas_pixel.shp')
print('Pixel shapefile saved')

# --- Shower Falls SNOTEL point ---
snotel_point = Point(point_lon, point_lat)
snotel_gdf = gpd.GeoDataFrame(
    {'name': ['Shower_Falls_SNOTEL'], 'geometry': [snotel_point]},
    crs='EPSG:4326'
)
snotel_gdf.to_file(r'RESULTS\spatial\shower_falls_snotel.shp')
print('SNOTEL point shapefile saved')

Pixel shapefile saved
SNOTEL point shapefile saved


In [3]:
# Check pixel calculation step by step
print(f'Point longitude: {point_lon}')
print(f'Upper left longitude: {ul_lon}')
print(f'res_x: {res_x:.8f} degrees')

# How many pixels from the left edge?
col_exact = (point_lon - ul_lon) / res_x
row_exact = (ul_lat - point_lat) / res_y

print(f'\nExact column (float): {col_exact:.6f}')
print(f'Exact row (float):    {row_exact:.6f}')

# Integer pixel index
col = int(col_exact)
row = int(row_exact)
print(f'\nPixel column index: {col}')
print(f'Pixel row index:    {row}')

# Pixel corners
pixel_west  = ul_lon + col * res_x
pixel_east  = pixel_west + res_x
pixel_north = ul_lat - row * res_y
pixel_south = pixel_north - res_y

print(f'\nPixel west:  {pixel_west:.8f}')
print(f'Point lon:   {point_lon:.8f}')
print(f'Pixel east:  {pixel_east:.8f}')
print(f'\nPixel north: {pixel_north:.8f}')
print(f'Point lat:   {point_lat:.8f}')
print(f'Pixel south: {pixel_south:.8f}')

# Is the point actually on the corner or inside?
print(f'\nPoint is {(point_lon - pixel_west) / res_x * 100:.2f}% across pixel in x')
print(f'Point is {(pixel_north - point_lat) / res_y * 100:.2f}% down pixel in y')

Point longitude: -110.95
Upper left longitude: -124.73333333333333
res_x: 0.00833333 degrees

Exact column (float): 1654.000000
Exact row (float):    897.000000

Pixel column index: 1654
Pixel row index:    897

Pixel west:  -110.95000000
Point lon:   -110.95000000
Pixel east:  -110.94166667

Pixel north: 45.40000000
Point lat:   45.40000000
Pixel south: 45.39166667

Point is 0.00% across pixel in x
Point is 0.00% down pixel in y


In [1]:
#Pixel size investigation
import os, tarfile, gzip, shutil, calendar, requests, re, time
import rasterio
import rasterstats
from rasterio.transform import from_origin
from datetime import datetime
from pathlib import Path
from shapely.geometry import mapping, Point
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# --- Directories ---
data_dir = Path(r'C:\Users\jonas\OneDrive\Documents\MSU\2026 Spring\Data Science\Final Project\Data\SNODAS')
data_dir.mkdir(exist_ok=True, parents=True)

# --- Spatial parameters ---
ul_lat, ul_lon = 52.87500000000000, -124.73333333333333
lr_lat, lr_lon = 24.95000000000000, -66.94166666666667
nrows, ncols = 3351, 6935
res_x = (lr_lon - ul_lon) / ncols
res_y = (ul_lat - lr_lat) / nrows
transform = from_origin(ul_lon, ul_lat, res_x, res_y)
crs = 'EPSG:4326'
nodata = -9999

# --- HDR content (ENVI format) ---
hdr_content = """ENVI
samples = 6935
lines = 3351
bands = 1
header offset = 0
file type = ENVI Standard
data type = 2
interleave = bsq
byte order = 1
"""

# --- Shower Falls point ---
lat = 45.4
lon = -110.95
point_geom = mapping(Point(lon, lat))

# --- URL builders ---
def date_url(date: datetime) -> str:
    url = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/"
    url += f"{date.year}/{date.month:02}_{calendar.month_abbr[date.month]}"
    url += f"/SNODAS_{date.strftime('%Y%m%d')}.tar"
    return url

def urls_for_date_range(start_date: datetime, end_date: datetime) -> list:
    dates = pd.date_range(start_date, end_date, freq='1D')
    return [date_url(date) for date in dates]

# --- Network Retry Loop  ---
class DownloadFailedError(Exception):
    pass

def download_with_retry(url: str, out_path: Path, max_retries: int = 5):
    time.sleep(1) #small delay
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=60)
            with open(out_path, 'wb') as f:
                f.write(response.content)
            return True
        except Exception as e:
            print(f'Attempt {attempt} failed: {e}')
            if out_path.exists():
                os.remove(out_path)
            time.sleep(5 * attempt)
    raise DownloadFailedError(f'Failed after {max_retries} attempts: {url}')

# --- Main pipeline ---
# Thread-safe list for collecting rows
rows_lock = threading.Lock()
rows = []

def process_date(url: str) -> dict:
    filename = url.split('/')[-1]
    tar_path = data_dir / filename

    # 1. Download tar
    print(f'Downloading: {filename}')
    success = download_with_retry(url, tar_path)
    if not success:
        return None

    # 2. Extract 11034 .dat.gz
    with tarfile.open(tar_path, 'r') as tar:
        swe_member = next(
            (m for m in tar.getmembers() if '11034' in m.name and m.name.endswith('.dat.gz')),
            None
        )
        if swe_member is None:
            print(f'No SWE file found in {filename}')
            os.remove(tar_path)
            return None
        tar.extract(swe_member, path=data_dir)

    dat_gz_path = data_dir / swe_member.name

    # 3. Decompress .dat.gz to .dat
    dat_path = dat_gz_path.with_suffix('')
    with gzip.open(dat_gz_path, 'rb') as gz_in:
        with open(dat_path, 'wb') as dat_out:
            shutil.copyfileobj(gz_in, dat_out)

    # 4. Write .hdr
    hdr_path = dat_path.with_suffix('.hdr')
    with open(hdr_path, 'w') as f:
        f.write(hdr_content)

    # 5. Convert .dat to .tif
    tif_path = dat_path.with_suffix('.tif')
    with rasterio.open(dat_path) as src:
        data = src.read()
        width = src.width
        height = src.height
    with rasterio.open(
        tif_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=data.dtype,
        crs=crs, transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(data)

    # 6. Query SWE at Shower Falls
#    match = re.search(r'TS(\d{8})', dat_path.stem)
#    date = pd.to_datetime(match.group(1), format='%Y%m%d')
#    result = rasterstats.point_query(
#        [point_geom],
#        str(tif_path),
#        nodata=-9999,
#        interpolate='nearest'
#    )[0]
#    swe = result / 1000 if result is not None else None

    # 7. Delete all intermediates including .tif
#    for f in [tar_path, dat_gz_path, dat_path, hdr_path, tif_path]:
#        if f.exists():
#            os.remove(f)
#
#    print(f'Queried: {dat_path.stem} — SWE: {swe} m')
#    return {'date': date, 'swe_m': swe}

# --- Run for date range ---
urls = []
for year in range(2020, 2021):
    year_urls = urls_for_date_range(
        datetime(year, 3, 1),
        datetime(year, 3, 1)
    )
    urls.extend(year_urls)
    
start_time = time.time()
rows = []
failed_url = None

try:
    with ThreadPoolExecutor(max_workers=1) as executor:
        futures = {executor.submit(process_date, url): url for url in urls}
        for future in as_completed(futures):
            try:
                result = future.result()
                if result is not None:
                    rows.append(result)
            except DownloadFailedError as e:
                failed_url = futures[future]
                print(f'Download failed, stopping: {failed_url}')
                executor.shutdown(wait=False, cancel_futures=True)
                break
finally:
    # Always save whatever was collected
    if rows:
        df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
        out_csv = data_dir / 'SNODAS_swe_2004.csv'
        if out_csv.exists():
            existing_df = pd.read_csv(out_csv, parse_dates=['date'])
            df = pd.concat([existing_df, df]).sort_values('date').reset_index(drop=True)
        df.to_csv(out_csv, index=False)
        elapsed = time.time() - start_time
        print(f'Saved {len(df)} total records to {out_csv}')
        print(f'Elapsed time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
        if failed_url:
            print(f'Stopped at: {failed_url}')

Downloading: SNODAS_20200301.tar


C:\Users\jonas\miniconda3\envs\gphy471\lib\site-packages\rasterio\__init__.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
